In [1]:
import sys
print('Kernel Python:', sys.executable)
%pip install numpy pandas ipython ipywidgets

Kernel Python: c:\Users\Admin\Documents\GitHub\MachineLearning_TextModule\.venv\Scripts\python.exe
Note: you may need to restart the kernel to use updated packages.


# Final UI Notebook (Pipeline Controller)

Notebook này là giao diện chạy end-to-end cho dự án.

- UI và luồng thao tác nằm ở đây.
- Các file trong modules chỉ đóng vai trò backend functions.

Quy trình:
1. Setup môi trường + import module
2. Chọn cấu hình bằng widget
3. Load dữ liệu
4. Run pipeline và xem bảng kết quả

In [2]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
from dataclasses import replace

import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / "modules").exists() and (p / "requirements.txt").exists():
            return p
    raise RuntimeError("Cannot find project root")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

from modules.config import Config
from modules.data_loader import load_data
from modules.pipeline import build_features, run_evaluation

base_cfg = Config()
state = {
    "cfg": None,
    "result_df": None,
    "x_train": None,
    "x_test": None,
    "n_train": None,
    "n_test": None,
}

PROJECT_ROOT: c:\Users\Admin\Documents\GitHub\MachineLearning_TextModule


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## UI controls

Chọn mode/feature/model ngay trong notebook UI.

In [3]:
import ipywidgets as widgets
from IPython.display import display

mode_w = widgets.Dropdown(
    options=["demo", "full"],
    value=base_cfg.mode,
    description="Mode:",
    layout=widgets.Layout(width="220px"),
)

feature_w = widgets.Dropdown(
    options=["tfidf", "sbert"],
    value=base_cfg.feature_method,
    description="Feature:",
    layout=widgets.Layout(width="220px"),
)

model_w = widgets.Dropdown(
    options=["logistic_regression", "svm", "naive_bayes"],
    value=base_cfg.model_type,
    description="Model:",
    layout=widgets.Layout(width="280px"),
)

preprocess_w = widgets.Checkbox(
    value=base_cfg.use_preprocessing,
    description="Use preprocessing",
)

run_btn = widgets.Button(
    description="Run pipeline",
    button_style="success",
    tooltip="Build features + train + evaluate",
)

out = widgets.Output()

display(widgets.HBox([mode_w, feature_w, model_w]))
display(preprocess_w)
display(run_btn)
display(out)

Checkbox(value=True, description='Use preprocessing')

Button(button_style='success', description='Run pipeline', style=ButtonStyle(), tooltip='Build features + trai…

Output()

## Load dataset (from module backend)

Dữ liệu được tải qua modules.data_loader để thống nhất source of truth.

In [4]:
train_texts, train_labels, test_texts, test_labels, info = load_data(base_cfg.dataset_name)

print(info)
print("Train:", len(train_texts), "Test:", len(test_texts))
print("Sample text:", str(train_texts[0])[:160], "...")

DatasetInfo(name='ag_news', train_size=120000, test_size=7600, text_field='text', label_field='label', num_classes=4)
Train: 120000 Test: 7600
Sample text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again. ...


## Run pipeline from UI

Cell này gắn sự kiện cho nút Run và hiển thị bảng metrics ngay trong notebook.

In [5]:
def build_cfg_from_ui():
    return replace(
        base_cfg,
        mode=mode_w.value,
        feature_method=feature_w.value,
        model_type=model_w.value,
        use_preprocessing=preprocess_w.value,
    )

def on_run_click(_):
    out.clear_output()
    with out:
        cfg = build_cfg_from_ui()
        print("Running config:")
        print(f"- mode={cfg.mode}")
        print(f"- feature_method={cfg.feature_method}")
        print(f"- model_type={cfg.model_type}")
        print(f"- use_preprocessing={cfg.use_preprocessing}")

        X_train, X_test, n_train, n_test = build_features(cfg, train_texts, test_texts)
        result, df_metrics = run_evaluation(
            X_train,
            train_labels,
            X_test,
            test_labels,
            cfg,
            method_name=f"{cfg.feature_method}+{cfg.model_type}",
        )

        state["cfg"] = cfg
        state["x_train"] = X_train
        state["x_test"] = X_test
        state["n_train"] = n_train
        state["n_test"] = n_test
        state["result_df"] = df_metrics

        print(f"Used samples: train={n_train}, test={n_test}")
        print(f"Feature shapes: train={X_train.shape}, test={X_test.shape}")
        display(df_metrics)

run_btn.on_click(on_run_click)
print("UI callback is ready. Click 'Run pipeline' above.")

UI callback is ready. Click 'Run pipeline' above.


## Manual trigger (optional)

Dùng cell này nếu bạn muốn chạy nhanh mà không bấm nút.

In [6]:
# Optional: run once from current UI values
on_run_click(None)

## Output artifacts

Kiểm tra file output theo feature method đang chạy.

In [7]:
cfg = state.get("cfg")
if cfg is None:
    print("Chưa có kết quả. Hãy chạy pipeline trước.")
else:
    print("Last config:", cfg)
    if cfg.feature_method == "tfidf":
        print("TF-IDF train path:", cfg.tfidf_train_path)
        print("TF-IDF test path:", cfg.tfidf_test_path)
        print("Exists:", cfg.tfidf_train_path.exists(), cfg.tfidf_test_path.exists())
    else:
        print("SBERT train path:", cfg.bert_train_path)
        print("SBERT test path:", cfg.bert_test_path)
        print("Exists:", cfg.bert_train_path.exists(), cfg.bert_test_path.exists())

    if state.get("result_df") is not None:
        display(state["result_df"])

Last config: Config(dataset_name='ag_news', text_column='text', label_column='label', seed=42, test_size=0.2, n_train_demo=5000, n_test_demo=2000, n_train_emb_demo=2000, n_test_emb_demo=500, use_preprocessing=True, lowercase=True, remove_stopwords=True, remove_punctuation=True, remove_numbers=True, mode='demo', feature_method='sbert', max_features=10000, ngram_range=(1, 2), min_df=2, tfidf_train_name='tfidf_train.npy', tfidf_test_name='tfidf_test.npy', sbert_model_name='sentence-transformers/all-MiniLM-L6-v2', sbert_batch_size=32, sbert_normalize=True, use_sbert_preprocessing=False, bert_train_name='bert_train.npy', bert_test_name='bert_test.npy', model_type='logistic_regression', C=1.0, max_iter=200, epochs=5, learning_rate=2e-05)
SBERT train path: C:\Users\Admin\Documents\GitHub\MachineLearning_TextModule\features\bert\bert_train.npy
SBERT test path: C:\Users\Admin\Documents\GitHub\MachineLearning_TextModule\features\bert\bert_test.npy
Exists: True True


,accuracy,precision_weighted,recall_weighted,f1_weighted,method
0,0.85,0.851769,0.85,0.848682,sbert+logistic_regression


## Week 3 validation checklist

Cell này kiểm tra nhanh các artifacts bắt buộc của Tuần 3:
- Embedding theo từng scale benchmark (`5k_2k`, `20k_2k`)
- Bảng báo cáo `results/tables/bert_benchmark_results.csv`

Nếu thiếu, chạy 2 cell code ngay dưới để tạo lại đầy đủ.

In [ ]:
import subprocess
import sys

# Đặt True khi bạn muốn chạy benchmark Tuần 3 đầy đủ.
RUN_FULL_WEEK3_BENCHMARK = False

if RUN_FULL_WEEK3_BENCHMARK:
    cmd = [sys.executable, "bert_benchmark.py"]
    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=False)
    print("Exit code:", completed.returncode)
else:
    print("Skip run. Set RUN_FULL_WEEK3_BENCHMARK = True để chạy benchmark Tuần 3.")

In [ ]:
from pathlib import Path

project_root = find_project_root()
required_paths = [
    project_root / "features" / "bert" / "5k_2k" / "bert_train.npy",
    project_root / "features" / "bert" / "5k_2k" / "bert_test.npy",
    project_root / "features" / "bert" / "20k_2k" / "bert_train.npy",
    project_root / "features" / "bert" / "20k_2k" / "bert_test.npy",
    project_root / "results" / "tables" / "bert_benchmark_results.csv",
]

missing = [p for p in required_paths if not p.exists()]

print("Week 3 artifact check:")
for p in required_paths:
    print(f"- {p.relative_to(project_root)}: {'OK' if p.exists() else 'MISSING'}")

if missing:
    print("\n=> Chưa đủ artifacts Tuần 3. Chạy cell benchmark bên dưới để tạo lại.")
else:
    print("\n=> Đã đủ artifacts Tuần 3.")

## Final Demo: Agency End-to-End Workflow

Phần này là demo cuối cho buổi trình bày:
1. Chọn objective và chạy workflow one-entry.
2. Đọc Critic verdict và Reporter recommendation.
3. Hiển thị bảng so sánh cuối cùng để chốt mô hình.

In [ ]:
import subprocess
import sys
import json
from pathlib import Path

# Objective options: fast | balanced | best
AGENCY_OBJECTIVE = "best"
RUN_AGENCY_DEMO = False

if RUN_AGENCY_DEMO:
    cmd = [sys.executable, "scripts/run_agency_workflow.py", AGENCY_OBJECTIVE]
    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=False)
    print("Exit code:", completed.returncode)
else:
    print("Skip run. Set RUN_AGENCY_DEMO = True để chạy demo agency.")

In [ ]:
critic_path = PROJECT_ROOT / "results" / "logs" / "critic_report.json"
summary_md_path = PROJECT_ROOT / "results" / "reports" / "agency_summary.md"
summary_json_path = PROJECT_ROOT / "results" / "reports" / "agency_summary.json"

print("Critic report:", critic_path)
if critic_path.exists():
    critic_data = json.loads(critic_path.read_text(encoding="utf-8"))
    print("- status:", critic_data.get("status"))
    print("- best_family:", critic_data.get("best_family"))
    print("- best_model:", critic_data.get("best_model"))
    print("- best_metric:", critic_data.get("best_metric"))
    findings = critic_data.get("findings", [])
    if findings:
        print("- findings:")
        for f in findings:
            print("  *", f.get("severity"), f.get("code"), "-", f.get("message"))
    else:
        print("- findings: none")
else:
    print("- missing critic report. Run agency workflow cell above first.")

print("\nReporter summary markdown:", summary_md_path)
print("Exists:", summary_md_path.exists())
print("Reporter summary json:", summary_json_path)
print("Exists:", summary_json_path.exists())

In [ ]:
import pandas as pd
from IPython.display import display

best_cmp_path = PROJECT_ROOT / "results" / "tables" / "feature_family_comparison.csv"
full_cmp_path = PROJECT_ROOT / "results" / "tables" / "feature_family_comparison_full.csv"

print("Best-vs-best comparison:", best_cmp_path)
if best_cmp_path.exists():
    df_best_cmp = pd.read_csv(best_cmp_path)
    display(df_best_cmp)
else:
    print("- missing file. Run agency workflow cell above first.")

print("\nScale-aware comparison:", full_cmp_path)
if full_cmp_path.exists():
    df_full_cmp = pd.read_csv(full_cmp_path)
    display(df_full_cmp)
else:
    print("- missing file. Run agency workflow cell above first.")